# Milestone 1 — Dataset and deterministic evaluation foundation

## Goal

Build a trustworthy, reproducible, and leakage-safe measurement foundation before introducing any language model.

At this checkpoint we have completed the **dataset-freezing portion** of Milestone 1. We have not yet built the result-normalization and SQL-result comparison logic.

### Mental model

A model benchmark is an exam:

- FINCH supplies questions and an intended SQL answer key.
- `working_dev` is the visible practice set used to develop the system.
- `frozen_eval` is the sealed final exam used for fair comparisons.
- The answer key must be audited before model performance can be trusted.

This notebook records how the slice was selected, how reference-data problems were discovered, how narrowly scoped compatibility rules were validated, and how the final evaluation manifest was frozen.


## 0. Environment

This notebook expects the local FINCH metadata at `../data/finch_dataset.json` and the BIRD `financial.sqlite` database at `../data/financial.sqlite`. Both files stay outside Git because the repository stores reproducible setup logic and metadata rather than redistributing the dataset.


In [ ]:
import sys

## 1. Inventory the FINCH financial slice

FINCH contains 75,725 natural-language/SQL pairs across several financial databases. We narrow the first experiment to the BIRD `financial` database so the schema and failure modes remain understandable.

The inventory establishes the available population before we select anything:

- 106 examples belong to BIRD `financial`.
- All 106 carry FINCH's upstream `dev` label, so DomainAdapt must define its own internal roles.
- Difficulty distribution: 62 easy, 37 medium, and 7 hard.
- IDs 1–3 were viewed during project orientation and are excluded from later selection.


In [ ]:
from collections import Counter
from pathlib import Path
import json

dataset_path = Path("../data/finch_dataset.json")
records = json.loads(dataset_path.read_text(encoding="utf-8"))

financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ids = [row["question_id"] for row in financial]

print("Total FINCH records:", len(records))
print("BIRD financial records:", len(financial))
print("Partition counts:", dict(Counter(row["partition"] for row in financial)))
print("Difficulty counts:", dict(Counter(row["difficulty"] for row in financial)))
print("Question ID range:", min(ids), "to", max(ids))
print("Unique question IDs:", len(set(ids)))
print("Known orientation IDs present:", sorted(set(ids) & {1, 2, 3}))

## 2. Create reproducible working and frozen sets

Think of the benchmark as an exam:

| Internal role | Size | Easy | Medium | Hard | Use |
|---|---:|---:|---:|---:|---|
| `working_dev` | 12 | 6 | 4 | 2 | Visible practice set for evaluator, prompt, and harness development |
| `frozen_eval` | 24 | 12 | 9 | 3 | Hidden exam used consistently across model and system variants |

Selection is deterministic rather than hand-picked. Each eligible question receives a SHA-256 rank derived from a fixed salt, its difficulty, and its ID. Anyone using the same files and rules will reproduce the same candidates.

The file hashes act as fingerprints for the exact FINCH JSON and SQLite database used. If an upstream file changes, validation will fail instead of silently changing the benchmark.

### Leakage boundary

- `working_dev` may be inspected, but it cannot be used as training or few-shot data.
- `frozen_eval` questions, SQL, and results remain unopened during development.
- All 106 examples from this database are excluded from future SFT, synthetic-training prompts, and few-shot pools. Later training examples must come from other databases.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import sqlite3

dataset_path = Path("../data/finch_dataset.json").resolve()
db_path = Path("../data/financial.sqlite").resolve()

records = json.loads(dataset_path.read_text(encoding="utf-8"))
financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ORIENTATION_IDS = {1, 2, 3}
SELECTION_SALT = "domain-adapt-financial-v1"

split_spec = {
    "working_dev": {"easy": 6, "medium": 4, "hard": 2},
    "frozen_eval": {"easy": 12, "medium": 9, "hard": 3},
}

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def selection_rank(row: dict) -> str:
    value = (
        f"{SELECTION_SALT}:"
        f"{row['difficulty']}:"
        f"{row['question_id']}"
    )
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

eligible = [
    row for row in financial
    if row["question_id"] not in ORIENTATION_IDS
]

by_difficulty = defaultdict(list)
for row in eligible:
    by_difficulty[row["difficulty"]].append(row)

for difficulty in by_difficulty:
    by_difficulty[difficulty].sort(key=selection_rank)

selected = {"working_dev": [], "frozen_eval": []}
offsets = Counter()

for role, difficulty_counts in split_spec.items():
    for difficulty, count in difficulty_counts.items():
        start = offsets[difficulty]
        stop = start + count
        selected[role].extend(by_difficulty[difficulty][start:stop])
        offsets[difficulty] = stop

print("Dataset SHA-256:", file_sha256(dataset_path))
print("Database SHA-256:", file_sha256(db_path))

for role, rows in selected.items():
    print(f"\n{role}:")
    print("  total:", len(rows))
    print("  difficulty:", dict(Counter(r["difficulty"] for r in rows)))
    print("  IDs:", sorted(r["question_id"] for r in rows))

## 3. Audit the supplied answer key before trusting it

FINCH supplies reference SQL—the benchmark's intended answer key. Successful execution alone does not prove that a query is meaningful: an incorrect filter can still execute and return an empty result.

This first audit flags mechanically suspicious outcomes:

- execution errors;
- no result rows;
- any `NULL` value;
- a single numeric zero.

These signals are not automatically errors. They tell us which examples require investigation before entering a trusted benchmark.


In [ ]:
def audit_reference(row: dict, conn: sqlite3.Connection) -> dict:
    try:
        cursor = conn.execute(row["SQL"])
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

conn = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

for role, rows in selected.items():
    audits = [audit_reference(row, conn) for row in rows]
    flagged = [
        audit
        for audit in audits
        if not audit["executes"] or audit["concerns"]
    ]

    print(f"\n{role}:")
    print("  executed:", sum(a["executes"] for a in audits), "/", len(audits))
    print("  flagged:", flagged)

## 4. Diagnose suspicious examples using only `working_dev`

We may inspect the visible working set without contaminating the hidden evaluation set. The following cell prints only flagged working examples so we can compare the question, reference SQL, and observed result.

The initial audit found seven suspicious working queries. Their pattern suggested a systematic incompatibility between FINCH's SQL and the supplied SQLite database rather than seven unrelated semantic mistakes.


In [ ]:
working_audits = {
    row["question_id"]: audit_reference(row, conn)
    for row in selected["working_dev"]
}

flagged_working = [
    row
    for row in selected["working_dev"]
    if working_audits[row["question_id"]]["concerns"]
]

for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    audit = working_audits[row["question_id"]]
    result_preview = conn.execute(row["SQL"]).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Difficulty:", row["difficulty"])
    print("Question:", row["question"])
    print("Reference SQL:", row["SQL"])
    print("Result preview:", result_preview)
    print("Concern:", audit["concerns"])

## 5. Confirm the database representations

The diagnostic queries revealed two benchmark compatibility problems:

1. **Categorical value casing.** FINCH reference SQL uses literals such as `'m'`, `'owner'`, and `'c'`, while the database stores `M`, `OWNER`, and `C`. SQLite's default equality comparison treats those as different values.
2. **Invalid SQLite year formatting.** FINCH uses `STRFTIME('%y', date)`, but SQLite does not support `%y`. It returns `NULL`. The database stores ISO-style dates, and `%Y` correctly returns their four-digit year.

These are answer-key/database mismatches. They must be resolved before any model is scored.


In [ ]:
diagnostics = {
    "client.gender": """
        SELECT gender, COUNT(*)
        FROM client
        GROUP BY gender
        ORDER BY gender
    """,
    "loan.status": """
        SELECT status, COUNT(*)
        FROM loan
        GROUP BY status
        ORDER BY status
    """,
    "disp.type": """
        SELECT type, COUNT(*)
        FROM disp
        GROUP BY type
        ORDER BY type
    """,
    "account.frequency": """
        SELECT frequency, COUNT(*)
        FROM account
        GROUP BY frequency
        ORDER BY frequency
    """,
    "North Bohemia spelling": """
        SELECT DISTINCT A3
        FROM district
        WHERE LOWER(A3) = 'north bohemia'
    """,
    "account date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM account
        LIMIT 5
    """,
    "loan date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM loan
        LIMIT 5
    """,
}

for label, sql in diagnostics.items():
    print(f"\n{label}:")
    print(conn.execute(sql).fetchall())

## 6. Define explicit reference compatibility rules

We introduce two narrow, documented transformations:

- replace unsupported `STRFTIME('%y', ...)` with `STRFTIME('%Y', ...)`;
- apply `COLLATE NOCASE` to reference-SQL equality comparisons against string literals.

The transformation applies **only to FINCH reference SQL** when producing expected results. Model-generated SQL will later execute exactly as generated against the untouched database. The evaluator therefore does not repair model mistakes.

This regex implementation is exploratory notebook logic. Once the behavior stabilizes, it can be replaced by a more robust reusable implementation.


In [ ]:
import re

def make_reference_compatible(sql: str) -> tuple[str, list[str]]:
    compatible = sql
    applied_rules = []

    # FINCH uses a four-digit year literal but SQLite does not support %y.
    updated = re.sub(
        r"STRFTIME\(\s*'%y'\s*,",
        "STRFTIME('%Y',",
        compatible,
        flags=re.IGNORECASE,
    )
    if updated != compatible:
        applied_rules.append("sqlite_four_digit_year")
        compatible = updated

    # FINCH lowercases categorical literals while the database retains casing.
    updated = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)?)"
        r"\s*=\s*('(?:''|[^'])*')",
        r"\1 = \2 COLLATE NOCASE",
        compatible,
    )
    if updated != compatible:
        applied_rules.append("case_insensitive_text_equality")
        compatible = updated

    return compatible, applied_rules

### Validate the repairs on flagged working examples

Re-running the seven visible failures gives meaningful results after the compatibility rules. Examples changed from zero or `NULL` to substantive results, confirming that the rules address systematic dataset incompatibilities.


In [ ]:
for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])
    result = conn.execute(compatible_sql).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Applied rules:", applied_rules)
    print("Compatible result:", result)

### Re-audit the complete working set

The compatible audit records which transformations were applied and repeats the same suspicious-result checks. All 12 working references must pass before the fixed rules are allowed to touch hidden candidates.


In [ ]:
def audit_compatible_reference(
    row: dict,
    conn: sqlite3.Connection,
) -> dict:
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])

    try:
        cursor = conn.execute(compatible_sql)
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "applied_rules": applied_rules,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "applied_rules": applied_rules,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

working_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["working_dev"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in working_compatible_audits),
    "/",
    len(working_compatible_audits),
)

print(
    "Still flagged:",
    [
        audit
        for audit in working_compatible_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

## 7. Audit frozen candidates without opening them

The rules were learned entirely from `working_dev`. We now apply them mechanically to frozen candidates while printing only IDs and audit metadata—never question text, SQL, or result values.

Two candidates remained ambiguous after compatibility repair: IDs 37 and 104. We do not inspect them because doing so would expose part of the hidden exam.


In [ ]:
frozen_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

frozen_flagged = [
    audit
    for audit in frozen_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in frozen_compatible_audits),
    "/",
    len(frozen_compatible_audits),
)
print("Still flagged:", frozen_flagged)

print(
    "Compatibility-rule usage:",
    dict(
        Counter(
            rule
            for audit in frozen_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

### Check the wider eligible pool

Across all 103 non-orientation examples, every compatible reference query executes and only three remain mechanically ambiguous. This shows that the two compatibility rules explain nearly all of the observed mismatch rather than merely fitting the selected working examples.


In [ ]:
all_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in eligible
]

all_flagged = [
    audit
    for audit in all_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "All eligible executed:",
    sum(audit["executes"] for audit in all_compatible_audits),
    "/",
    len(all_compatible_audits),
)
print("All eligible still flagged:", len(all_flagged))

print(
    "Remaining concern types:",
    dict(
        Counter(
            concern
            for audit in all_flagged
            for concern in audit["concerns"]
        )
    ),
)

print(
    "Compatibility-rule usage across eligible pool:",
    dict(
        Counter(
            rule
            for audit in all_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

## 8. Replace ambiguous candidates conservatively

For this small V1 benchmark, a frozen reference must:

- execute successfully;
- return at least one row;
- contain no `NULL`;
- avoid a singleton numeric zero.

This conservative policy reduces the chance that an incorrect model query receives credit because both it and the reference accidentally return nothing.

IDs 37 and 104 are removed without inspection. Deterministic ranking selects clean replacements of the same difficulty: IDs 31 and 85.


In [ ]:
audit_by_id = {
    audit["question_id"]: audit
    for audit in all_compatible_audits
}

def is_clean_reference(question_id: int) -> bool:
    audit = audit_by_id[question_id]
    return audit["executes"] and not audit["concerns"]

working_ids = {
    row["question_id"]
    for row in selected["working_dev"]
}

replacement_frozen = []

for difficulty, required_count in split_spec["frozen_eval"].items():
    candidates = [
        row
        for row in by_difficulty[difficulty]
        if row["question_id"] not in working_ids
        and is_clean_reference(row["question_id"])
    ]

    replacement_frozen.extend(candidates[:required_count])

old_frozen_ids = {
    row["question_id"]
    for row in selected["frozen_eval"]
}
new_frozen_ids = {
    row["question_id"]
    for row in replacement_frozen
}

print("Removed:", sorted(old_frozen_ids - new_frozen_ids))
print("Added:", sorted(new_frozen_ids - old_frozen_ids))
print("Final frozen IDs:", sorted(new_frozen_ids))
print(
    "Final difficulty counts:",
    dict(Counter(row["difficulty"] for row in replacement_frozen)),
)

selected["frozen_eval"] = replacement_frozen

### Validate the final frozen set

The final set must contain exactly 24 clean references with the intended 12 easy, 9 medium, and 3 hard distribution. Assertions turn these benchmark assumptions into executable checks.


In [ ]:
final_frozen_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

print("Total:", len(final_frozen_audits))
print(
    "Executed:",
    sum(audit["executes"] for audit in final_frozen_audits),
)
print(
    "Flagged:",
    [
        audit
        for audit in final_frozen_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

assert len(final_frozen_audits) == 24
assert all(audit["executes"] for audit in final_frozen_audits)
assert all(not audit["concerns"] for audit in final_frozen_audits)

## 9. Freeze the benchmark manifest

The manifest is the sealed inventory for `domain-adapt-financial-v1`. It stores:

- exact source-file hashes;
- database and upstream partition identity;
- deterministic selection rules and salt;
- working and frozen IDs;
- orientation and ambiguous-reference exclusions;
- reference compatibility rules;
- audit requirements;
- leakage policy.

It records IDs and provenance rather than copying FINCH questions or SQL into the repository.


In [ ]:
manifest = {
    "manifest_version": 1,
    "benchmark": "domain-adapt-financial-v1",
    "created_on": "2026-09-25",
    "source": {
        "dataset_repository": "domyn/FINCH",
        "dataset_file": "finch_dataset.json",
        "dataset_sha256": file_sha256(dataset_path),
        "database_file": "text2sql-db/text2sql/bird/financial.sqlite",
        "database_sha256": file_sha256(db_path),
        "license": "CC-BY-NC-4.0",
    },
    "scope": {
        "db_name": "bird",
        "db_id": "financial",
        "upstream_partition": "dev",
    },
    "selection": {
        "algorithm": (
            "SHA-256 rank within difficulty using "
            "'{salt}:{difficulty}:{question_id}'"
        ),
        "salt": SELECTION_SALT,
        "orientation_exclusions": sorted(ORIENTATION_IDS),
        "ambiguous_reference_exclusions": sorted(
            audit["question_id"]
            for audit in all_flagged
        ),
        "working_dev": {
            "ids": sorted(
                row["question_id"]
                for row in selected["working_dev"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["working_dev"]
                )
            ),
        },
        "frozen_eval": {
            "ids": sorted(
                row["question_id"]
                for row in selected["frozen_eval"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["frozen_eval"]
                )
            ),
        },
    },
    "reference_compatibility": {
        "scope": "reference_sql_only",
        "rules": [
            {
                "id": "sqlite_four_digit_year",
                "description": (
                    "Replace unsupported SQLite STRFTIME('%y', ...) "
                    "with STRFTIME('%Y', ...)."
                ),
            },
            {
                "id": "case_insensitive_text_equality",
                "description": (
                    "Apply COLLATE NOCASE to reference-SQL equality "
                    "comparisons against string literals because FINCH "
                    "lowercases literals while the database retains casing."
                ),
            },
        ],
    },
    "reference_audit": {
        "requirements": [
            "SQL executes successfully",
            "result contains at least one row",
            "result contains no NULL values",
            "result is not a singleton numeric zero",
        ],
    },
    "leakage_policy": {
        "working_dev": (
            "May be inspected for evaluator, prompt, and harness development; "
            "must not be used as few-shot or training data."
        ),
        "frozen_eval": (
            "Questions, SQL, and results must remain uninspected during "
            "system development and must never be used for prompting, "
            "few-shot selection, synthetic generation, or training."
        ),
        "financial_database": (
            "All 106 FINCH BIRD financial examples are excluded from "
            "few-shot pools and model-training data."
        ),
    },
}

manifest_path = Path("../configs/eval_manifest_v1.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

print(manifest_path.resolve())
print(manifest_path.read_text(encoding="utf-8"))

### Validate the saved manifest

The final assertions verify source fingerprints, split sizes, and disjointness. If any of these change, the benchmark is no longer the frozen V1 evaluation set.


In [ ]:
loaded_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert loaded_manifest["source"]["dataset_sha256"] == (
    "c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130"
)
assert loaded_manifest["source"]["database_sha256"] == (
    "d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e"
)
assert len(loaded_manifest["selection"]["working_dev"]["ids"]) == 12
assert len(loaded_manifest["selection"]["frozen_eval"]["ids"]) == 24
assert set(
    loaded_manifest["selection"]["working_dev"]["ids"]
).isdisjoint(
    loaded_manifest["selection"]["frozen_eval"]["ids"]
)

print("Frozen manifest validated.")

## Checkpoint summary

We started with 106 FINCH examples for one understandable banking database and produced:

- a visible 12-question `working_dev` set;
- a hidden 24-question `frozen_eval` set;
- deterministic selection and exact file provenance;
- database-level protection against training and few-shot leakage;
- two evidence-backed reference compatibility rules;
- a conservative audit that excludes ambiguous reference results;
- a committed manifest that reproduces the benchmark.

The SQLite database remains unchanged. Compatibility repair applies only to FINCH's reference SQL. Model-generated SQL will receive no repair.

## Next step

Build deterministic result comparison:

1. execute compatible reference SQL;
2. execute candidate/model SQL;
3. normalize rows and scalar values;
4. account for meaningful ordering and floating-point tolerance;
5. decide whether both queries produced an equivalent result;
6. prove that every frozen reference passes the evaluator against itself.
